# Notebook 01 — Attention by hand

Paired with `theory/01_scaled_dot_product.md`. **Mode:** theory-first — read the chapter end-to-end and write the §1.7 predictions before running anything.

## Question, Hypothesis, Method, Prediction, Confidence

**Question.** Implement scaled dot-product attention from the definition and verify the core structural claims of Chapter 1:
1. The worked-example output from §1.4 reproduces to floating-point precision.
2. Perfect-match retrieval: scaling the query concentrates attention on the matching key — how fast (entropy vs. $\lambda$)?
3. Equal keys: attention is forced to uniform and the output collapses to the value mean.
4. Self-attention on a toy sequence shows content-based grouping as an off-diagonal attention pattern.
5. The closed-form softmax Jacobian $\partial p_i / \partial s_j = p_i(\delta_{ij} - p_j)$ matches autograd (Prop 2.2 preview).
6. Argmax has zero gradient almost everywhere — the structural failure of hard attention (Prop 1.4).

**Hypothesis.** All six follow from the definition plus basic softmax properties. Chapter 1 is about the mental model; this notebook is confirmation. Anything that surprises exposes a gap in the model — note it.

**Method.** Six small experiments in numpy + torch, no training.
1. Implement `attention(Q, K, V)` from scratch; reproduce §1.4 numerically.
2. Orthogonal keys + query-scale sweep $\lambda \in \{0.1, 1, 10, 100\}$; measure attention entropy and top weight.
3. Equal-keys case: confirm uniform weights and value-mean output.
4. Self-attention on a handcrafted 5-token sequence; visualize the $5 \times 5$ attention matrix.
5. Autograd Jacobian of softmax vs. closed form over many random score vectors.
6. Gradient norms for three variants: standard softmax, near-hard softmax ($T = 0.01$), and argmax (finite differences).

**Prediction.** *Paste your §1.7 predictions here. For each: direction, order-of-magnitude estimate, confidence.*

1. Worked-example floating-point error — 
2. $\lambda$ at which attention on key 2 crosses 99% — 
3. Equal-keys output — 
4. Toy-sequence attention pattern — 
5. Jacobian max-abs difference — 
6. Argmax gradient norm — 

**Confidence.** Overall: *[low / medium / high, one-line reason]*.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from attn_lab import plot_attention, plot_softmax_bars, get_device

np.random.seed(0)
torch.manual_seed(0)
np.set_printoptions(precision=4, suppress=True)
device = get_device()
print('device:', device)

---
## Experiment 1 — implementation + worked example (§1.4)

Implement `attention(Q, K, V)` from the definition, then verify the hand-computed outputs $(5, 5)$ and $\approx (3.30, 6.70)$.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # shift invariance (Prop 2.1)
    ex = np.exp(x)
    return ex / ex.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)       # (n, m)
    weights = softmax(scores, axis=-1)    # (n, m)
    output = weights @ V                  # (n, d_v)
    return output, weights

Q = np.array([[1., 0.], [0., 1.]])
K = np.array([[1., 0.], [1., 1.]])
V = np.array([[10., 0.], [0., 10.]])

out, w = attention(Q, K, V)
print('Attention weights:')
print(w)
print('\nOutput:')
print(out)
print('\nExpected: [[5.0, 5.0], [~3.302, ~6.698]]')
print(f'Row 1 max-abs error: {np.max(np.abs(out[0] - np.array([5.0, 5.0]))):.2e}')
print(f'Row 2 max-abs error: {np.max(np.abs(out[1] - np.array([3.302, 6.698]))):.2e}')

**Sub-finding 1.** *1–2 sentences. Did the output match to floating-point precision? Any numerical surprises in the attention weights?*

---
## Experiment 2 — perfect-match retrieval under query scaling

Four orthogonal keys, query equal to key 2 scaled by $\lambda \in \{0.1, 1, 10, 100\}$. Measure attention entropy and the weight placed on the matching key.

This is the Exercise 1.1 setup. Since scores scale linearly with $\lambda$ (query magnitude) and softmax concentration is exponential in score margin, we expect rapid concentration.

In [ ]:
d_k = 8
n = 4
K = np.eye(n, d_k)                  # 4 orthogonal unit keys
V = np.arange(1, n + 1).reshape(-1, 1).astype(float) * np.ones((1, 3))  # row j = [j, j, j]
q_base = K[2].copy()                # query matches key 2

scales = [0.1, 1.0, 10.0, 100.0]
print(f'{"lambda":>8}  {"weight on k_2":>14}  {"entropy (nats)":>16}  {"output[0]":>12}')
print('-' * 56)
for lam in scales:
    q = lam * q_base
    _, w = attention(q.reshape(1, -1), K, V)
    w_row = w[0]
    entropy = -(w_row * np.log(w_row + 1e-30)).sum()
    out = w_row @ V
    print(f'{lam:>8.1f}  {w_row[2]:>14.6f}  {entropy:>16.4f}  {out[0]:>12.4f}')

In [ ]:
# Fine-grained sweep: find the lambda at which weight on k_2 crosses 0.99.
lambdas = np.linspace(0.0, 5.0, 200)
top_weights = []
entropies = []
for lam in lambdas:
    q = lam * q_base
    _, w = attention(q.reshape(1, -1), K, V)
    top_weights.append(w[0, 2])
    entropies.append(-(w[0] * np.log(w[0] + 1e-30)).sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lambdas, top_weights)
axes[0].axhline(0.99, color='r', linestyle=':', label='0.99 threshold')
axes[0].set_xlabel('lambda (query scale)')
axes[0].set_ylabel('weight on key 2')
axes[0].set_title('Concentration vs query scale')
axes[0].legend()

axes[1].plot(lambdas, entropies)
axes[1].axhline(np.log(n), color='k', linestyle=':', alpha=0.5, label=f'log({n}) = uniform')
axes[1].set_xlabel('lambda (query scale)')
axes[1].set_ylabel('attention entropy (nats)')
axes[1].set_title('Entropy vs query scale')
axes[1].legend()
plt.tight_layout()
plt.show()

# Locate the crossing.
idx = int(np.searchsorted(top_weights, 0.99))
if idx < len(lambdas):
    print(f'Attention on k_2 first crosses 0.99 at lambda ≈ {lambdas[idx]:.3f}  (score margin ≈ {lambdas[idx] / np.sqrt(d_k):.3f})')

**Sub-finding 2.** *At which $\lambda$ did attention cross 99% on key 2? Does the entropy curve decay polynomially or exponentially in $\lambda$? Match against your §1.7 prediction.*

---
## Experiment 3 — equal keys

All keys equal to a single vector $\mathbf{k}^\star$. Scores are all equal, softmax gives uniform weights, and output = column-wise mean of $V$.

In [ ]:
d_k = 8
n = 5
k_star = np.random.randn(d_k)
K_equal = np.tile(k_star, (n, 1))
V_random = np.random.randn(n, 3)
q = np.random.randn(d_k)

out, w = attention(q.reshape(1, -1), K_equal, V_random)
print('Attention weights (should be uniform = 0.2 each):')
print(w[0])
print('\nOutput:')
print(out[0])
print('Column means of V:')
print(V_random.mean(axis=0))
print(f'\nMax abs deviation from mean: {np.max(np.abs(out[0] - V_random.mean(axis=0))):.2e}')

**Sub-finding 3.** *Uniform attention + value mean confirmed? What does this say about the behavior of attention early in training, when $W_K$ is near zero and keys are nearly identical?*

---
## Experiment 4 — self-attention on a handcrafted sequence

Five tokens. Tokens 0 and 3 share signal A; tokens 1 and 4 share signal B; token 2 is orthogonal. Self-attention with $W_Q = W_K = W_V = I$ should group tokens by signal — visible as off-diagonal concentration.

In [ ]:
rng = np.random.default_rng(0)
d = 16

e_A = rng.normal(size=d)
e_B = rng.normal(size=d)
e_C = rng.normal(size=d)
noise = lambda: 0.05 * rng.normal(size=d)
X = np.stack([
    e_A + noise(),   # t0 — signal A
    e_B + noise(),   # t1 — signal B
    e_C + noise(),   # t2 — orthogonal
    e_A + noise(),   # t3 — signal A
    e_B + noise(),   # t4 — signal B
])

_, W = attention(X, X, X)
fig, ax = plt.subplots(figsize=(5, 4))
plot_attention(W, xlabels=['t0', 't1', 't2', 't3', 't4'], ylabels=['t0', 't1', 't2', 't3', 't4'],
               title='self-attention weights  (Q = K = V = X)', ax=ax)
plt.show()

# Quantitative check: does row 0 concentrate most mass on {0, 3}?
print(f'Row 0 mass on {{t0, t3}}: {W[0, [0, 3]].sum():.3f}')
print(f'Row 1 mass on {{t1, t4}}: {W[1, [1, 4]].sum():.3f}')
print(f'Row 2 mass on t2:         {W[2, 2]:.3f}')

**Sub-finding 4.** *Did rows 0 and 3 concentrate on each other (signal-A group)? Rows 1 and 4 (signal-B group)? What did row 2 do, and what does that say about tokens with no matching peers in the sequence?*

---
## Experiment 5 — softmax Jacobian vs. autograd (Prop 2.2 preview)

$\partial p_i / \partial s_j = p_i(\delta_{ij} - p_j)$. Compute closed form and compare against autograd over many random score vectors.

In [ ]:
n = 5
n_trials = 50
max_diffs = []

for _ in range(n_trials):
    s = torch.randn(n, requires_grad=True)
    p = F.softmax(s, dim=-1)
    J_autograd = torch.autograd.functional.jacobian(lambda x: x.softmax(-1), s)
    J_formula = torch.diag(p) - p.unsqueeze(1) * p.unsqueeze(0)
    max_diffs.append((J_autograd - J_formula).abs().max().item())

max_diffs = np.array(max_diffs)
print(f'Max-abs difference across {n_trials} trials: {max_diffs.max():.2e}')
print(f'Mean-abs difference:                     {max_diffs.mean():.2e}')

# Row-sum check: Exercise 2.2 says each row of J sums to zero.
print(f'\nLast-trial row sums (should be ~0): {J_formula.sum(dim=1).detach().numpy()}')

**Sub-finding 5.** *Did the closed form match autograd to floating-point precision? Did rows sum to zero? This confirms Prop 2.2 and the mass-conservation property from Exercise 2.2.*

---
## Experiment 6 — argmax has zero gradient almost everywhere

Compare gradient norms across three pseudo-attention variants over a scalar loss $\ell = \langle p(s), t \rangle$ for random $t$:

- (i) standard softmax, $T = 1$.
- (ii) near-hard softmax, $T = 0.01$. Saturated for moderate score spreads.
- (iii) argmax (one-hot at the max). Non-differentiable; gradient measured via finite differences.

Prediction from theory: (i) moderate, well-conditioned; (ii) very small (softmax saturated, Jacobian near zero, Lemma 3.3 preview); (iii) exactly zero for almost all $s$ — argmax is piecewise constant, so the FD gradient is zero except on the measure-zero boundary where the argmax switches.

In [ ]:
n = 10
n_trials = 500
T_hard = 0.01
target = torch.randn(n)

grad_norms = {'softmax (T=1)': [], 'near-hard (T=0.01)': [], 'argmax': []}

for _ in range(n_trials):
    s_base = torch.randn(n)

    # (i) Standard softmax.
    s1 = s_base.clone().requires_grad_(True)
    loss1 = (F.softmax(s1, dim=-1) * target).sum()
    g1 = torch.autograd.grad(loss1, s1)[0]
    grad_norms['softmax (T=1)'].append(g1.norm().item())

    # (ii) Near-hard softmax.
    s2 = s_base.clone().requires_grad_(True)
    loss2 = (F.softmax(s2 / T_hard, dim=-1) * target).sum()
    g2 = torch.autograd.grad(loss2, s2)[0]
    grad_norms['near-hard (T=0.01)'].append(g2.norm().item())

    # (iii) Argmax via finite differences.
    s3 = s_base.numpy()
    def argmax_loss(x):
        onehot = np.zeros(n)
        onehot[x.argmax()] = 1.0
        return float(onehot @ target.numpy())
    eps = 1e-4
    fd = np.zeros(n)
    base_loss = argmax_loss(s3)
    for j in range(n):
        s3p = s3.copy(); s3p[j] += eps
        fd[j] = (argmax_loss(s3p) - base_loss) / eps
    grad_norms['argmax'].append(float(np.linalg.norm(fd)))

print(f'{"variant":>22}   {"mean |grad|":>14}   {"median |grad|":>16}   {"frac zero":>10}')
print('-' * 68)
for name, vals in grad_norms.items():
    vals = np.array(vals)
    print(f'{name:>22}   {vals.mean():>14.4e}   {np.median(vals):>16.4e}   {(vals == 0).mean():>10.2%}')

**Sub-finding 6.** *Was the argmax gradient zero almost always? How much smaller was near-hard softmax's gradient vs. standard softmax? Does this match your §1.7 prediction and set up the Ch. 3 story (saturation → vanishing gradients)?*

---
## Finding

*Write 3–6 sentences:*

- Which predictions held?
- Which missed, and in what direction? State the mental-model update in one sentence.
- Any surprises (unexpected concentration speed, attention-row structure, precision artifacts)?
- What's the next question this raises? Candidate: how does the $\lambda$-scaling argument from Experiment 2 reappear when we scale $d_k$ (Chapter 3) instead of the query?

*Append 2–3 sentences to `learnings.md` with today's date.*